# First step of this project is to reproduce Project Tycho's measles report from 1963.
- Dataset: https://www.tycho.pitt.edu/dataset/US.14189004/?utm_source=chatgpt.com
- Article: https://academic.oup.com/ofid/article/5/7/ofy137/5039595

In [1]:
core_columns = [
    "ConditionName",
    "ConditionSNOMED",
    "Admin1ISO",
    "Admin1Name",
    "Admin2Name",
    "PeriodStartDate",
    "PeriodEndDate",
    "CountValue",
    "PartOfCumulativeCountSeries",
    "Fatalities"
]

In [ ]:
import pandas as pd

measles_df = pd.read_csv('raw/tycho/US.14189004.csv', low_memory=False)[core_columns]
measles_df.head()

,ConditionName,ConditionSNOMED,Admin1ISO,Admin1Name,Admin2Name,PeriodStartDate,PeriodEndDate,CountValue,PartOfCumulativeCountSeries,Fatalities
0,Measles,14189004,US-WI,WISCONSIN,NaN,1927-11-20,1927-11-26,85,0,0
1,Measles,14189004,US-WI,WISCONSIN,NaN,1927-11-27,1927-12-03,120,0,0
2,Measles,14189004,US-WI,WISCONSIN,NaN,1927-12-04,1927-12-10,84,0,0
3,Measles,14189004,US-WI,WISCONSIN,NaN,1927-12-18,1927-12-24,106,0,0
4,Measles,14189004,US-WI,WISCONSIN,NaN,1927-12-25,1927-12-31,39,0,0


In [ ]:
measles_df['Year'] = measles_df['PeriodStartDate'].str.split('-').str[0].astype(int)
measles_df['State'] = measles_df['Admin1ISO'].str.split('-').str[1]

In [ ]:
measles_df.head()

,ConditionName,ConditionSNOMED,Admin1ISO,Admin1Name,Admin2Name,PeriodStartDate,PeriodEndDate,CountValue,PartOfCumulativeCountSeries,Fatalities,Year,State
0,Measles,14189004,US-WI,WISCONSIN,NaN,1927-11-20,1927-11-26,85,0,0,1927,WI
1,Measles,14189004,US-WI,WISCONSIN,NaN,1927-11-27,1927-12-03,120,0,0,1927,WI
2,Measles,14189004,US-WI,WISCONSIN,NaN,1927-12-04,1927-12-10,84,0,0,1927,WI
3,Measles,14189004,US-WI,WISCONSIN,NaN,1927-12-18,1927-12-24,106,0,0,1927,WI
4,Measles,14189004,US-WI,WISCONSIN,NaN,1927-12-25,1927-12-31,39,0,0,1927,WI


In [ ]:
plot_measles_df = measles_df[(measles_df['Year'] >= 1960) & (measles_df['Year'] <= 1970)]
plot_measles_df.head()

,ConditionName,ConditionSNOMED,Admin1ISO,Admin1Name,Admin2Name,PeriodStartDate,PeriodEndDate,CountValue,PartOfCumulativeCountSeries,Fatalities,Year,State
1592,Measles,14189004,US-WI,WISCONSIN,NaN,1960-01-03,1960-01-09,410,0,0,1960,WI
1593,Measles,14189004,US-WI,WISCONSIN,NaN,1960-01-10,1960-01-16,663,0,0,1960,WI
1594,Measles,14189004,US-WI,WISCONSIN,NaN,1960-01-17,1960-01-23,429,0,0,1960,WI
1595,Measles,14189004,US-WI,WISCONSIN,NaN,1960-01-24,1960-01-30,442,0,0,1960,WI
1596,Measles,14189004,US-WI,WISCONSIN,NaN,1960-01-31,1960-02-06,414,0,0,1960,WI


In [ ]:
import plotly.express as px

def animate_map(df):
    fig = px.choropleth(
        df,
        locations="State",
        locationmode="USA-states",
        color="CountValue",
        animation_frame="Year",
        scope="usa",
        color_continuous_scale="Reds",
        range_color=(df["CountValue"].min(), df["CountValue"].max()),
        title="Disease Incidence Over Time"
    )

    fig.show()

## Let's reproduce this paper: https://academic.oup.com/ofid/article/5/7/ofy137/5039595
Here's Table 1 as a markdown table:

**Table 1. Observed and Prevented Measles Cases, Deaths, and Related Costs in the United States, With 80% Uncertainty Range**

| | Prevaccination (1931–1963) | Introduction (1964–1970) | 1-dose Vaccine (1971–1989) | 2-dose Vaccine (1990–2014) | Entire Period (1964–2014) |
|---|---|---|---|---|---|
| **Cases, millions** | | | | | |
| Observed<sup>a</sup> | 16.81 | 1.14 | 0.39 | 0.05 | 1.57 |
| Prevented | — | 2.49 (0.30 to 8.35) | 10.12 (3.19 to 31.89) | 17.17 (5.59 to 57.60) | 29.78 (9.08 to 97.84) |
| **Deaths, thousands** | | | | | |
| Observed | 45.52 | 1.19 | 0.28 | 0.11 | 1.59 |
| Prevented | — | 1.46 (–1.18 to 10.50) | 10.51 (–0.28 to 76.48) | 19.61 (–0.11 to 334.61) | 31.57 (–1.57 to 421.59) |

<sup>a</sup>Observed cases, as reported by the US Centers for Disease Control and Prevention (CDC; data from Project Tycho and the CDC).
<sup>b</sup>Estimated costs due to hospitalization or lost income associated with reported measles cases based on state-level cost estimates.

### ***Note that project Tycho is between 1931 and 1992***
- Alaska and Hawaii are excluded
- Not coutning sub-state categories
- 32% weekly missing values - current dataset is pre-imuptation
- Only consider incident counts: PartOfCumulativeCountSeries==0

In [ ]:
exclude_regions = [
    'US-AS',  # American Samoa
    'US-GU',  # Guam
    'US-MP',  # Northern Mariana Islands
    'US-PR',  # Puerto Rico
    'US-VI',  # U.S. Virgin Islands
    'US-HI', 
    'US-AK'
]

essential_cols = ['PeriodStartDate', 'State', 'CountValue', 'Year']

repro_measles_df = measles_df[(measles_df['Year'] >= 1931) 
                                & (measles_df['Year'] <= 1992) 
                                & (measles_df['PartOfCumulativeCountSeries'] == 0) 
                                & (~measles_df['Admin1ISO'].isin(exclude_regions))
                                & (measles_df['Admin2Name'].isna())
                              ].drop_duplicates()

repro_measles_df = repro_measles_df[essential_cols]
repro_measles_df['PeriodStartDate'] = pd.to_datetime(repro_measles_df['PeriodStartDate'])

len(repro_measles_df)

102968

In [ ]:
prevaccination = repro_measles_df[(repro_measles_df['Year'] >= 1931) & (repro_measles_df['Year'] <= 1963)]
print(prevaccination['CountValue'].sum())

15878288


In [ ]:
introduction = repro_measles_df[(repro_measles_df['Year'] >= 1964) & (repro_measles_df['Year'] <= 1970)]
print(introduction['CountValue'].sum())

977089


In [ ]:
one_dose_vaccine = repro_measles_df[(repro_measles_df['Year'] >= 1971) & (repro_measles_df['Year'] <= 1989)]
print(one_dose_vaccine['CountValue'].sum())

316373


### The case counts are close but are not exact as we have to consider the article's situation on 
- 32% weekly missing values, study used imputation - current dataset is pre-imputation

***Let's simulate this imputation to get closer results***
- As stated by the article:
    - We imputed missing weekly Project Tycho counts for each state with the average of counts for the preceding and following weeks for which data were available (linear imputation).

Here is how ChatGPT explains it:

missing week → average of preceding and following available weeks.
Example:
- Week 1: 10 cases
- Week 3: ?? cases
- Week 2: 20 cases

To calculate week 3, we take the average of week 1 and 3. so now:
- Week 1: 10 cases
- Week 3: 15 cases
- Week 2: 20 cases

### Note that 49 states were in this study, refer to ```exclude_regions```

Here is our method
- identify missing weeks for each state and insert new rows with the missing week and CountValue as NaN
- use pandas interpolate to fill in the missing values

In [ ]:
missing_dates = pd.DataFrame(columns=essential_cols)
for state in repro_measles_df['State'].unique(): # iterate through reproducible dataset for each state

    state_df = repro_measles_df[repro_measles_df['State'] == state].sort_values(by='PeriodStartDate')

    for i in range(len(state_df)):
        current_row = state_df.iloc[i]

        # Next row
        if i + 1 < len(state_df):
            next_row = state_df.iloc[i + 1]
        else:
            next_row = None

        # we only start imputation if current row is not the first or the last
        if next_row is not None:
            
            # first we check that the next row is the following week from the current row
            day_difference = next_row["PeriodStartDate"] - current_row["PeriodStartDate"]
            if day_difference.days != 7:

                # generate the missing dates
                new_dates = pd.date_range(
                    start=current_row["PeriodStartDate"] + pd.Timedelta(days=7),
                    end=next_row["PeriodStartDate"] - pd.Timedelta(days=7),
                    freq="7D"
                )

                count_val = (current_row['CountValue'] + next_row['CountValue']) / 2
                for date in new_dates:

                    row_dict = {
                        'PeriodStartDate': pd.Timestamp(date),
                        'State':current_row['State'],
                        'CountValue': count_val,
                        'Year': current_row['Year']
                        }
                    
                    pd_series = pd.Series(row_dict)
                    missing_dates = pd.concat([missing_dates, pd_series.to_frame().T], ignore_index=True)

In [ ]:
repro_measles_df = pd.concat([repro_measles_df, missing_dates], ignore_index=True)

### Here we can see what % was missing, as the paper states 32%


In [ ]:
len(missing_dates) / len(repro_measles_df)

0.32879641218189404

### Now we can interpolate!

In [ ]:
prevaccination = repro_measles_df[(repro_measles_df['Year'] >= 1931) & (repro_measles_df['Year'] <= 1963)]
print(round(prevaccination['CountValue'].sum(), 2))

16766215.5


In [ ]:
introduction = repro_measles_df[(repro_measles_df['Year'] >= 1964) & (repro_measles_df['Year'] <= 1970)]
print(round(introduction['CountValue'].sum(), 2))

1032533.5


In [ ]:
one_dose_vaccine = repro_measles_df[(repro_measles_df['Year'] >= 1971) & (repro_measles_df['Year'] <= 1989)]
print(round(one_dose_vaccine['CountValue'].sum(), 2))

438387.0
